# MoE Throughput Sweep — Challenge 2

Three sections:
- **B** — 3B architecture: dense vs MoE variants at fixed topology (4n EP=4)
- **C** — 3B topology: EP sweep (EP=2/4/8/16) and node scaling (1n–8n)
- **D** — 8B validation: do 3B conclusions hold at larger scale?

Fill in W&B run IDs after each section completes. Run IDs are the short hash at the end of the W&B run URL.

In [2]:
import os
import wandb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

In [5]:
api = wandb.Api()
runs = api.runs(
    "LSAIE/gipfelsturm",
    filters={"username": "tobiasuruali"},
    order="-created_at",
)

for run in runs:
    print(f"{run.name:70s}  {run.state}")

throughput-3b-4n-moe64top4ep2-ffn2048-moe-3b-topo-2256631               failed
throughput-3b-4n-moe64top4ep4-ffn2048-moe-3b-topo-2256630               crashed
throughput-3b-4n-moe128top4ep8-ffn2048-moe-3b-arch-2256629              crashed
throughput-3b-4n-moe64top4ep4-ffn2048-lf2-moe-3b-arch-2256628           crashed
throughput-3b-4n-moe64top8ep4-ffn1024-moe-3b-arch-2256627               failed
throughput-3b-4n-moe64top4ep4-ffn2048-moe-3b-arch-2256626               crashed
throughput-3b-2n-moe64top4ep4-ffn2048-moe-3b-topo-2256635               failed
throughput-3b-4n-moe64top2ep4-ffn4096-moe-3b-arch-2256625               failed
throughput-3b-1n-moe-3b-topo-2256634                                    finished
throughput-3b-4n-moe8top2ep4-ffn4096-moe-3b-arch-2256624                failed
throughput-3b-4n-moe-3b-arch-2256623                                    finished
throughput-3b-4n-moe8top2ep4-ffn4096-moe-3b-arch-2256405                failed
throughput-760m-1n-moe64top4ep4-ffn1024-moe-

In [2]:
# ── Plotting constants (match data-parallel notebook style) ──────────────────
DOUBLE_WIDTH_PT = 487.8225
DOUBLE_COL_WIDTH_INCH = DOUBLE_WIDTH_PT / 72.27
SMALL_FONT = 10
TICK_FONT = 8

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": SMALL_FONT,
    "axes.labelsize": SMALL_FONT,
    "axes.titlesize": SMALL_FONT,
    "xtick.labelsize": TICK_FONT,
    "ytick.labelsize": TICK_FONT,
})

ENTITY  = "LSAIE"
PROJECT = "gipfelsturm"
STEADY_STATE_START = 40   # steps used for steady-state average
STEADY_STATE_END   = 50

In [3]:
# ── W&B run ID registry ───────────────────────────────────────────────────────
# Format: "LSAIE/gipfelsturm/<run_id>"
# Find the run_id in the W&B URL or from the wandb summary line at the end of each log.

# Section B — 3B architecture (project: moe-3b-arch)
ARCH_RUNS = {
    "[3] dense":          None,  # TODO: fill in e.g. "LSAIE/gipfelsturm/abc12345"
    "[4] 8E top-2":       None,
    "[5] 64E top-2":      None,
    "[6] 64E top-4":      None,  # main candidate
    "[7] 64E top-8":      None,
    "[8] 64E top-4 lf=2": None,
    "[9] 128E top-4":     None,
}

# Section C — 3B topology (project: moe-3b-topo)
# EP sweep (4 nodes fixed)
EP_RUNS = {
    2:  None,   # [10] EP=2
    4:  None,   # [6'] EP=4 anchor
    8:  None,   # [11] EP=8
    16: None,   # [12] EP=16
}
# Node scaling (EP=4 fixed)
NODE_RUNS = {
    1:  None,   # [13] dense 1n reference
    2:  None,   # [14] MoE 2n
    4:  None,   # [6'] MoE 4n (same run as EP_RUNS[4])
    8:  None,   # [15] MoE 8n
}

# Section D — 8B validation (project: moe-8b-val)
VAL_RUNS = {
    "[16] dense":          None,
    "[17] 8E top-2":       None,
    "[18] 16E top-4":      None,
    "[19] 16E top-4 lf=2": None,
    "[20] 64E top-4 EP=8": None,
}

In [4]:
# ── Fetch helper ──────────────────────────────────────────────────────────────
def fetch_steady_state(run_path):
    """Return mean/std of tokens-per-sec-per-gpu over steady-state steps."""
    api = wandb.Api()
    run = api.run(run_path)
    history = run.history()
    subset = history[
        (history["_step"] >= STEADY_STATE_START) &
        (history["_step"] <= STEADY_STATE_END)
    ]
    col = "tokens-per-sec-per-gpu"
    return {
        "mean": subset[col].mean(),
        "std":  subset[col].std(),
    }


def build_df(run_dict, key_name):
    """Fetch all runs in a dict {key: run_path} and return a DataFrame."""
    rows = []
    for key, path in run_dict.items():
        if path is None:
            print(f"  skipping {key} — no run ID yet")
            continue
        print(f"  fetching {key} …")
        m = fetch_steady_state(path)
        rows.append({key_name: key, "mean": m["mean"], "std": m["std"]})
    return pd.DataFrame(rows)

## Section B — 3B Architecture

In [5]:
ARCH_CSV = "moe_arch_metrics.csv"

if os.path.exists(ARCH_CSV):
    arch_df = pd.read_csv(ARCH_CSV)
else:
    print("Fetching Section B runs from W&B …")
    arch_df = build_df(ARCH_RUNS, key_name="config")
    arch_df.to_csv(ARCH_CSV, index=False)

arch_df

Fetching Section B runs from W&B …
  skipping [3] dense — no run ID yet
  skipping [4] 8E top-2 — no run ID yet
  skipping [5] 64E top-2 — no run ID yet
  skipping [6] 64E top-4 — no run ID yet
  skipping [7] 64E top-8 — no run ID yet
  skipping [8] 64E top-4 lf=2 — no run ID yet
  skipping [9] 128E top-4 — no run ID yet


""


In [ ]:
# ── Plot B: horizontal bar chart, normalised to dense baseline ────────────────
if not arch_df.empty and "[3] dense" in arch_df["config"].values:
    dense_mean = arch_df.loc[arch_df["config"] == "[3] dense", "mean"].values[0]
    arch_df["rel_mean"] = arch_df["mean"] / dense_mean
    arch_df["rel_std"]  = arch_df["std"]  / dense_mean

    config_order = list(ARCH_RUNS.keys())
    plot_df = arch_df.set_index("config").reindex(config_order).dropna(subset=["rel_mean"])

    fig, ax = plt.subplots(figsize=(DOUBLE_COL_WIDTH_INCH * 0.6, 3.0))

    colors = ["#d62728" if c == "[3] dense" else "#1f77b4" for c in plot_df.index]
    bars = ax.barh(
        plot_df.index, plot_df["rel_mean"],
        xerr=plot_df["rel_std"], capsize=3,
        color=colors, height=0.6, error_kw={"linewidth": 0.8}
    )
    ax.axvline(1.0, color="#d62728", linewidth=0.8, linestyle="--", alpha=0.7)
    ax.set_xlabel("Throughput relative to 3B dense", fontsize=SMALL_FONT)
    ax.tick_params(axis="both", labelsize=TICK_FONT)
    ax.grid(True, axis="x", alpha=0.3)
    ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig("moe_arch_sweep.pdf", bbox_inches="tight")
    plt.show()

## Section C — 3B Topology

In [ ]:
EP_CSV   = "moe_ep_metrics.csv"
NODE_CSV = "moe_node_metrics.csv"

if os.path.exists(EP_CSV):
    ep_df = pd.read_csv(EP_CSV)
else:
    print("Fetching EP sweep runs from W&B …")
    ep_df = build_df(EP_RUNS, key_name="ep")
    ep_df.to_csv(EP_CSV, index=False)

if os.path.exists(NODE_CSV):
    node_df = pd.read_csv(NODE_CSV)
else:
    print("Fetching node scaling runs from W&B …")
    node_df = build_df(NODE_RUNS, key_name="nodes")
    node_df.to_csv(NODE_CSV, index=False)

ep_df, node_df

In [ ]:
# ── Plot C: EP sweep + node scaling side by side ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL_WIDTH_INCH, 2.5))

# EP sweep
if not ep_df.empty:
    ep_plot = ep_df.sort_values("ep")
    axes[0].plot(ep_plot["ep"], ep_plot["mean"], marker="o", markersize=4, linewidth=1.2, color="#1f77b4")
    axes[0].fill_between(
        ep_plot["ep"],
        ep_plot["mean"] - ep_plot["std"],
        ep_plot["mean"] + ep_plot["std"],
        alpha=0.15, color="#1f77b4"
    )
    # Mark NVLink / Slingshot boundary
    axes[0].axvspan(0, 4.5, alpha=0.05, color="green", label="NVLink A2A")
    axes[0].axvspan(4.5, 20, alpha=0.05, color="red",   label="Slingshot A2A")
    axes[0].set_xlabel("Expert parallelism (EP)", fontsize=SMALL_FONT)
    axes[0].set_ylabel("Tokens / sec / GPU", fontsize=SMALL_FONT)
    axes[0].set_xscale("log", base=2)
    axes[0].set_xticks([2, 4, 8, 16])
    axes[0].xaxis.set_major_formatter(mticker.ScalarFormatter())
    axes[0].tick_params(axis="both", labelsize=TICK_FONT)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=TICK_FONT, framealpha=0.0)

# Node scaling
if not node_df.empty:
    node_plot = node_df.sort_values("nodes")
    # Distinguish dense 1n from MoE nodes
    moe_nodes = node_plot[node_plot["nodes"] > 1]
    dense_1n  = node_plot[node_plot["nodes"] == 1]
    axes[1].plot(moe_nodes["nodes"], moe_nodes["mean"], marker="o", markersize=4,
                 linewidth=1.2, color="#1f77b4", label="MoE EP=4")
    axes[1].fill_between(
        moe_nodes["nodes"],
        moe_nodes["mean"] - moe_nodes["std"],
        moe_nodes["mean"] + moe_nodes["std"],
        alpha=0.15, color="#1f77b4"
    )
    if not dense_1n.empty:
        axes[1].axhline(dense_1n["mean"].values[0], color="#d62728",
                        linewidth=0.8, linestyle="--", label="dense 1n ref")
    axes[1].set_xlabel("Nodes", fontsize=SMALL_FONT)
    axes[1].set_ylabel("Tokens / sec / GPU", fontsize=SMALL_FONT)
    axes[1].set_xscale("log", base=2)
    axes[1].set_xticks([1, 2, 4, 8])
    axes[1].xaxis.set_major_formatter(mticker.ScalarFormatter())
    axes[1].tick_params(axis="both", labelsize=TICK_FONT)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=TICK_FONT, framealpha=0.0)

plt.tight_layout()
plt.savefig("moe_topology_sweep.pdf", bbox_inches="tight")
plt.show()

## Section D — 8B Validation

In [ ]:
VAL_CSV = "moe_8b_val_metrics.csv"

if os.path.exists(VAL_CSV):
    val_df = pd.read_csv(VAL_CSV)
else:
    print("Fetching Section D runs from W&B …")
    val_df = build_df(VAL_RUNS, key_name="config")
    val_df.to_csv(VAL_CSV, index=False)

val_df

In [ ]:
# ── Plot D: same bar chart style as Section B ─────────────────────────────────
if not val_df.empty and "[16] dense" in val_df["config"].values:
    dense_mean = val_df.loc[val_df["config"] == "[16] dense", "mean"].values[0]
    val_df["rel_mean"] = val_df["mean"] / dense_mean
    val_df["rel_std"]  = val_df["std"]  / dense_mean

    config_order = list(VAL_RUNS.keys())
    plot_df = val_df.set_index("config").reindex(config_order).dropna(subset=["rel_mean"])

    fig, ax = plt.subplots(figsize=(DOUBLE_COL_WIDTH_INCH * 0.6, 2.5))

    colors = ["#d62728" if c == "[16] dense" else "#1f77b4" for c in plot_df.index]
    ax.barh(
        plot_df.index, plot_df["rel_mean"],
        xerr=plot_df["rel_std"], capsize=3,
        color=colors, height=0.6, error_kw={"linewidth": 0.8}
    )
    ax.axvline(1.0, color="#d62728", linewidth=0.8, linestyle="--", alpha=0.7)
    ax.set_xlabel("Throughput relative to 8B dense", fontsize=SMALL_FONT)
    ax.tick_params(axis="both", labelsize=TICK_FONT)
    ax.grid(True, axis="x", alpha=0.3)
    ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig("moe_8b_val.pdf", bbox_inches="tight")
    plt.show()

## Summary Table (LaTeX)

In [ ]:
# ── Combined LaTeX table for the report ──────────────────────────────────────
def fmt(mean, std):
    if pd.isna(mean): return "--"
    return rf"{int(mean):,} $\pm$ {int(std):,}"

print(r"\begin{tabular}{llrr}")
print(r"\toprule")
print(r"Section & Config & tok/s/GPU & Rel. to dense \\")
print(r"\midrule")

if not arch_df.empty:
    dense_mean = arch_df.loc[arch_df["config"]=="[3] dense", "mean"].values[0] if "[3] dense" in arch_df["config"].values else np.nan
    for _, row in arch_df.iterrows():
        rel = f"{row['mean']/dense_mean:.2f}" if not np.isnan(dense_mean) else "--"
        print(rf"B (3B arch) & {row['config']} & {fmt(row['mean'], row['std'])} & {rel} \\")
    print(r"\midrule")

if not val_df.empty:
    dense_mean = val_df.loc[val_df["config"]=="[16] dense", "mean"].values[0] if "[16] dense" in val_df["config"].values else np.nan
    for _, row in val_df.iterrows():
        rel = f"{row['mean']/dense_mean:.2f}" if not np.isnan(dense_mean) else "--"
        print(rf"D (8B val) & {row['config']} & {fmt(row['mean'], row['std'])} & {rel} \\")

print(r"\bottomrule")
print(r"\end{tabular}")